In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
!ls /content/drive/MyDrive/SARA

In [ ]:
!mkdir -p /content/SARA/dataset

In [ ]:
!unzip /content/drive/MyDrive/SARA/dataset2.zip -d /content/SARA/dataset

In [ ]:
!ls /content/SARA

In [ ]:
!mkdir /content/SARA/yolo

In [ ]:
from pathlib import Path

SOURCE=Path('/content/SARA/dataset')
OUTPUT=Path('/content/SARA/yolo')

In [ ]:
from pathlib import Path
import random
import shutil

random.seed(42)

# Find all images
images = sorted(SOURCE.glob("*.jpg"))

print("Images found:", len(images))

# Shuffle and split
random.shuffle(images)

split = int(len(images) * 0.8)

train_images = images[:split]
val_images = images[split:]

print("Training:", len(train_images))
print("Validation:", len(val_images))

# Create folders
for split_name in ["train", "val"]:
    (OUTPUT / split_name / "images").mkdir(parents=True, exist_ok=True)
    (OUTPUT / split_name / "labels").mkdir(parents=True, exist_ok=True)


# Function to copy images and labels
def copy_dataset(images, split_name):

    for image in images:

        # Copy image
        shutil.copy2(
            image,
            OUTPUT / split_name / "images" / image.name
        )

        # Check whether corresponding label exists
        label = SOURCE / f"{image.stem}.txt"

        if label.exists():
            # Copy label if available
            shutil.copy2(
                label,
                OUTPUT / split_name / "labels" / label.name
            )

        else:
            # No label = negative/background image
            print(f"Negative image (no objects): {image.name}")


# Copy training data
copy_dataset(train_images, "train")

# Copy validation data
copy_dataset(val_images, "val")

print("Dataset created at:", OUTPUT)

In [ ]:
yaml_content = """
path: /content/SARA/yolo

train: train/images
val: val/images

names:
  0: bottle_cap
"""

(OUTPUT / "data.yaml").write_text(yaml_content)

print((OUTPUT / "data.yaml").read_text())

In [ ]:
!pip install -U ultralytics

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.yaml")

In [ ]:
results = model.train(
    data="/content/SARA/yolo/data.yaml",
    epochs=100,
    imgsz=640,
    batch=32,
    device=0,
    workers=2,
    project="/content/runs",
    name="bottle_cap_scratch_100"
)

In [ ]:
metrics = model.val(
    data="/content/SARA/yolo/data.yaml",
    imgsz=640
)

print(metrics)

In [ ]:
import random
from pathlib import Path
from PIL import Image
from IPython.display import display

val_dir = Path("/content/SARA/yolo/val/images")

# Pick a random validation image
image_path = random.choice(list(val_dir.glob("*.jpg")))

# Predict
results = model.predict(
    source=str(image_path),
    conf=0.25,
    imgsz=640,
    verbose=False
)

# Display annotated image
annotated = results[0].plot()
display(Image.fromarray(annotated[:, :, ::-1]))

print("Image:", image_path.name)

In [ ]:
from google.colab import files

files.download("/content/runs/bottle_cap_scratch_100-5/weights/best.pt")